# MLE — Train Arcade Game RL Agent on Colab GPU

Runs N parallel MAME instances + trains PPO/DQN with GPU acceleration.

**Upload your ROM zip** (e.g. `frogger.zip`) when prompted.

In [ ]:
# 1. Install MAME + SDL + dependencies
!apt-get update -qq && apt-get install -y -qq mame xvfb > /dev/null 2>&1
!pip install -q gymnasium stable-baselines3 pillow

# Start virtual display for SDL
import subprocess, os
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1x1x24'])
os.environ['DISPLAY'] = ':99'
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'
os.environ['XDG_RUNTIME_DIR'] = '/tmp'

!which mame

In [ ]:
# 2. Clone MLE repo
!git clone https://github.com/lilwg/mle.git
%cd mle

In [ ]:
# 3. Upload ROM file
from google.colab import files
import os

uploaded = files.upload()

os.makedirs('roms', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'roms/{fname}')
    print(f'ROM: roms/{fname}')

rom_name = [f[:-4] for f in os.listdir('roms') if f.endswith('.zip')][0]
print(f'Game: {rom_name}')

In [ ]:
# 4. Quick test
import sys
sys.path.insert(0, '.')
from mle import MameEnv

env = MameEnv('roms', rom_name, {'_dummy': 0}, render=False, sound=False, throttle=False)
env.wait(100)
d = env.step()
print(f'MAME works! Got {len(d)} values')
env.close()
print('Ready to train!')

In [ ]:
# 5. Train with parallel MAME instances!
GAME = rom_name
TIMESTEPS = 2_000_000  # 2M steps
MODEL = 'ppo'
N_PARALLEL = 4          # 4 MAME instances = ~4x faster

save_name = f'{GAME}_{MODEL}_{TIMESTEPS//1000}k'
print(f'Training {GAME} with {MODEL}, {TIMESTEPS} steps, {N_PARALLEL} parallel envs')

!python3 mle_general.py {GAME} --model {MODEL} --timesteps {TIMESTEPS} \
    --save {save_name} --parallel {N_PARALLEL}

In [ ]:
# 6. Download trained model
from google.colab import files
import glob
for f in glob.glob('*.zip'):
    if 'cheat' not in f:
        print(f'Downloading {f}...')
        files.download(f)